# Análisis de uso del bot

Este notebook lee el registro de interacciones que genera el bot (`historial/historial.csv`) y mide **cómo se usa**: qué funciones se consultan más, quiénes lo usan, en qué momentos y por qué vía (comando, botón o texto libre).

**No hace falta tener el bot corriendo**: solo necesita que el CSV tenga datos. Cuantas más interacciones haya, más útiles son los gráficos, así que conviene usarlo un rato (entre varias personas) antes de analizar.

## Qué contiene el CSV

| Columna | Contenido |
|---|---|
| `timestamp` | Fecha y hora del mensaje (hora de la computadora donde corre el bot) |
| `chat_id` | Identificador del chat que lo envió |
| `nombre` | Nombre de quien lo envió |
| `mensaje` | Texto enviado, `[botón] ...` si tocó un botón, o `[audio]`, `[imagen]`, `[video]`, `[documento]`, `[sticker]` si no era texto |

> El archivo tiene nombres e identificadores reales, por eso **no se sube a GitHub** (está en el `.gitignore`).

## Dependencias

Se instalan una sola vez en el entorno virtual. La celda queda comentada para que no se reinstale cada vez.

In [ ]:
# %pip install pandas matplotlib
#%pip install pandas

## 1. Carga y limpieza de los datos

- Si el archivo no existe, el notebook corta con un mensaje claro.
- `chat_id` se lee como **texto** (no como número) porque es un identificador, no una cantidad.
- Los mensajes vacíos se reemplazan por texto vacío y las filas con fecha ilegible se descartan.
- `EXCLUIR_CHAT_IDS` permite sacar del análisis tus propias pruebas o las de otros integrantes del equipo: poné ahí los `chat_id` que no quieras contar.
- `GUARDAR_GRAFICOS` guarda cada gráfico como imagen en `archivos/img/png/`, útil para usarlos en el informe.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RUTA_CSV = Path("historial") / "historial.csv"
EXCLUIR_CHAT_IDS = [7067707562]          # ej: ["123456789"] para no contar tus pruebas
GUARDAR_GRAFICOS = True
CARPETA_GRAFICOS = Path("archivos") / "img" / "png"

if not RUTA_CSV.exists():
    raise FileNotFoundError(
        f"No encuentro {RUTA_CSV}. Ejecutá este notebook desde la carpeta del bot "
        "y usá el bot al menos una vez para que se genere el archivo."
    )

df = pd.read_csv(RUTA_CSV, encoding="utf-8", dtype={"chat_id": str})

columnas = {"timestamp", "chat_id", "nombre", "mensaje"}
if not columnas.issubset(df.columns):
    raise ValueError(f"El CSV debería tener las columnas {sorted(columnas)} y tiene {list(df.columns)}.")

df["mensaje"] = df["mensaje"].fillna("").astype(str)
df["nombre"] = df["nombre"].fillna("(sin nombre)").astype(str)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"])
df = df[~df["chat_id"].isin(EXCLUIR_CHAT_IDS)].reset_index(drop=True)

if df.empty:
    raise ValueError("No hay interacciones para analizar (el CSV está vacío o todo fue excluido).")

def guardar(fig, nombre):
    """Guarda el gráfico como PNG si GUARDAR_GRAFICOS está activo."""
    if GUARDAR_GRAFICOS:
        CARPETA_GRAFICOS.mkdir(parents=True, exist_ok=True)
        fig.savefig(CARPETA_GRAFICOS / f"{nombre}.png", dpi=150, bbox_inches="tight")

print(f"{len(df)} interacciones cargadas.")
df.head()

## 2. Clasificación de cada mensaje

Cada fila se clasifica de dos maneras:

- **`via`**: *cómo* lo pidió la persona: `Comando` (escribió `/dolar`), `Botón` (tocó un atajo del `/ayuda`), `Texto libre` (escribió algo que no es comando) o `No texto` (audio, imagen, video, documento, sticker).
- **`funcion`**: *qué* pidió. Un botón se asocia al comando equivalente (el botón 💵 cuenta como `/dolar`), así se puede medir cada función sin importar por qué vía se usó.

Los comandos que el bot no conoce (`/pepe`) se agrupan como `comando inexistente`.

In [ ]:
COMANDOS_CONOCIDOS = {"/start", "/ayuda", "/dolar", "/riesgo", "/presidente", "/inflacion", "/historial"}
BOTON_A_COMANDO = {
    "dolar": "/dolar",
    "riesgo_pais": "/riesgo",
    "presidentes": "/presidente",
    "inflacion": "/inflacion",
    "historial": "/historial",
}


def clasificar(mensaje: str):
    m = mensaje.strip()
    if m.startswith("[botón]"):
        accion = m.replace("[botón]", "").strip()
        return "Botón", BOTON_A_COMANDO.get(accion, "botón desconocido")
    if m.startswith("["):  # [audio], [imagen], [video], [documento], [sticker]...
        return "No texto", "mensaje no texto"
    if m.startswith("/"):
        comando = m.split()[0].split("@")[0].lower()
        return "Comando", comando if comando in COMANDOS_CONOCIDOS else "comando inexistente"
    return "Texto libre", "texto libre"


df[["via", "funcion"]] = df["mensaje"].apply(lambda m: pd.Series(clasificar(m)))
df["hora"] = df["timestamp"].dt.hour
df["fecha"] = df["timestamp"].dt.date
df[["timestamp", "nombre", "mensaje", "via", "funcion"]].tail(10)

## 3. Resumen general

In [ ]:
usuarios = df["chat_id"].nunique()
primera, ultima = df["timestamp"].min(), df["timestamp"].max()
dias = max((ultima.date() - primera.date()).days + 1, 1)

resumen = pd.Series({
    "Interacciones totales": len(df),
    "Usuarios distintos": usuarios,
    "Promedio por usuario": round(len(df) / usuarios, 1),
    "Primer registro": primera.strftime("%Y-%m-%d %H:%M"),
    "Último registro": ultima.strftime("%Y-%m-%d %H:%M"),
    "Días con datos (rango)": dias,
    "Interacciones por día (promedio)": round(len(df) / dias, 1),
}, name="Valor")
resumen.to_frame()

## 4. ¿Qué funciones se usan más?

Cuenta cada pedido por función, sumando comando y botón (y mostrando también los mensajes que el bot no pudo interpretar). Sirve para ver qué parte del bot le interesa a la gente y cuál casi no se toca.

In [ ]:
por_funcion = df["funcion"].value_counts()
display(por_funcion.to_frame("interacciones"))

fig, ax = plt.subplots(figsize=(8, 4.5))
por_funcion.sort_values().plot.barh(ax=ax, color="#4C78A8")
ax.bar_label(ax.containers[0], padding=3)
ax.set_title("Interacciones por función")
ax.set_xlabel("Cantidad")
ax.set_ylabel("")
ax.margins(x=0.1)
fig.tight_layout()
guardar(fig, "uso_por_funcion")
plt.show()

## 5. ¿Cómo lo piden? Comando, botón o texto libre

Mide si la gente prefiere los atajos del `/ayuda` o escribir los comandos, y cuánta escribe cosas que el bot no entiende (texto libre y mensajes no soportados). Un porcentaje alto de texto libre indica que los usuarios esperan una conversación y no solo comandos, una pista para los próximos pasos.

In [ ]:
por_via = df["via"].value_counts()
porcentaje = (por_via / por_via.sum() * 100).round(1)
display(pd.DataFrame({"interacciones": por_via, "porcentaje": porcentaje}))

fig, ax = plt.subplots(figsize=(7, 4))
por_via.plot.bar(ax=ax, color="#F58518", rot=0)
ax.bar_label(ax.containers[0], padding=3)
ax.set_title("Interacciones según la vía usada")
ax.set_xlabel("")
ax.set_ylabel("Cantidad")
ax.margins(y=0.12)
fig.tight_layout()
guardar(fig, "uso_por_via")
plt.show()

## 6. ¿A qué hora se usa?

Cantidad de interacciones por hora del día (0 a 23). Las horas sin actividad se muestran en cero para que el eje esté completo. La hora es la de la computadora donde corre el bot.

In [ ]:
por_hora = df["hora"].value_counts().reindex(range(24), fill_value=0).sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
por_hora.plot.bar(ax=ax, color="#54A24B", rot=0)
ax.set_title("Interacciones por hora del día")
ax.set_xlabel("Hora")
ax.set_ylabel("Cantidad")
fig.tight_layout()
guardar(fig, "uso_por_hora")
plt.show()

print(f"Hora más activa: {por_hora.idxmax()}:00 h ({por_hora.max()} interacciones)")

## 7. Evolución por día

In [ ]:
por_dia = df.groupby("fecha").size()

fig, ax = plt.subplots(figsize=(9, 4))
por_dia.plot(ax=ax, marker="o", color="#B279A2")
ax.set_title("Interacciones por día")
ax.set_xlabel("Fecha")
ax.set_ylabel("Cantidad")
ax.grid(alpha=0.3)
fig.tight_layout()
guardar(fig, "uso_por_dia")
plt.show()

print(f"Día más activo: {por_dia.idxmax()} ({por_dia.max()} interacciones)")

## 8. ¿Quiénes lo usan?

Ranking de usuarios por cantidad de interacciones (los 10 primeros). Se agrupa por `chat_id`, no por nombre, porque dos personas pueden llamarse igual.

In [ ]:
por_usuario = (
    df.groupby("chat_id")
    .agg(nombre=("nombre", "last"), interacciones=("mensaje", "size"))
    .sort_values("interacciones", ascending=False)
)
top = por_usuario.head(10)
display(top)

fig, ax = plt.subplots(figsize=(8, 4.5))
top.set_index("nombre")["interacciones"].sort_values().plot.barh(ax=ax, color="#E45756")
ax.bar_label(ax.containers[0], padding=3)
ax.set_title("Usuarios más activos")
ax.set_xlabel("Cantidad")
ax.set_ylabel("")
ax.margins(x=0.1)
fig.tight_layout()
guardar(fig, "uso_por_usuario")
plt.show()

## 9. ¿Qué usa cada persona?

Tabla cruzada de usuario por función: muestra si todos usan lo mismo o cada uno se enfoca en una función distinta.

In [ ]:
cruce = pd.crosstab(df["nombre"], df["funcion"])
cruce["TOTAL"] = cruce.sum(axis=1)
cruce.sort_values("TOTAL", ascending=False).head(10)